# 04 — Test the Full System

Step 15: run agent pipeline against several scenarios:
- Sad text + sad image → expected **High Risk**
- Happy text + happy image → expected **Stable**
- Mixed inputs → expected **Conflict / Mixed**

Also evaluates CNN on the held-out `data/img/test/` folder.

In [ ]:
from PIL import Image

from notebook_setup import add_project_to_path
from config import IMG_TEST
from system import System, format_report

add_project_to_path()
system = System()
print('DeepFace backend:', system.deepface.backend)

## Scenario tests

In [ ]:
def first_image(emotion: str) -> Image.Image:
    p = next((IMG_TEST / emotion).iterdir())
    return Image.open(p)

scenarios = [
    ('I feel empty, hopeless, and exhausted every single day.', 'sad', 'Sad + sad'),
    ('Had a wonderful day with my family, feeling really happy.', 'happy', 'Happy + happy'),
    ('I feel terrible and lonely, but the photo is from my birthday.', 'happy', 'Mixed (negative text, happy face)'),
]
for text, emo, label in scenarios:
    print('=' * 60)
    print('SCENARIO:', label)
    img = first_image(emo)
    res = system.analyze(text, img)
    print(format_report(res))
    print()

## CNN accuracy on FER test split

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from config import IMG_SIZE

tfm = transforms.Compose([transforms.Resize(IMG_SIZE), transforms.ToTensor()])
test_ds = datasets.ImageFolder(IMG_TEST, transform=tfm)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

model = system.cnn.model
device = system.cnn.device
model.eval()
correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
print(f'CNN test accuracy: {correct/max(1,total):.4f}  ({correct}/{total})')

## NLP sanity check

In [ ]:
samples = [
    'I feel anxious and cannot sleep at night.',
    'Such a productive and joyful day at work!',
    'Sometimes I am tired but mostly things are fine.',
]
for s in samples:
    r = system.nlp.predict(s)
    print(f'{r.short_label:9s}  conf={r.confidence:.2f}  | {s}')